# Task 1: Install required Python libraries

In [ ]:
%pip install -q pandas scikit-learn streamlit joblib requests

import pandas
import sklearn
import streamlit
import joblib
import requests

print(pandas.__version__)
print(sklearn.__version__)
print(streamlit.__version__)
print(joblib.__version__)
print(requests.__version__)

# Task 2: Download the dataset

In [2]:
%pip install -q pandas requests

import pandas as pd
import requests
from pathlib import Path

# Create data/ folder if it doesn't exist
Path("data").mkdir(exist_ok=True)

data_file = Path("data/results.csv")

# Download only if not already present
if not data_file.exists():
    url = (
        "https://raw.githubusercontent.com/IBM-SkillsBuild-AI-Builders-Challenge"
        "/hands-on-labs/main/02_football_lab_june/04_data/results.csv"
    )
    response = requests.get(url)
    response.raise_for_status()
    data_file.write_bytes(response.content)

# Load into DataFrame
matches = pd.read_csv(data_file, parse_dates=["date"])

print(matches.shape)
print(matches["date"].min())
print(matches["date"].max())
matches.head(3)

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
(49329, 9)
1872-11-30 00:00:00
2026-06-27 00:00:00


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False


# Task 3: Explore the data

In [3]:
import pandas as pd

# --- Top 10 most frequent tournaments ---
print("Top 10 tournaments by match count:")
print(matches["tournament"].value_counts().head(10).to_string())

# --- Top 15 teams by total matches played ---
home_counts = matches["home_team"].value_counts()
away_counts = matches["away_team"].value_counts()
total_matches = home_counts.add(away_counts, fill_value=0).astype(int)
print("\nTop 15 teams by total matches played:")
print(total_matches.sort_values(ascending=False).head(15).to_string())

# --- Matches per decade ---
decade = (matches["date"].dt.year // 10 * 10).map(lambda y: f"{y}s")
print("\nMatches per decade:")
print(decade.value_counts().sort_index().to_string())

Top 10 tournaments by match count:
tournament
Friendly                                18257
Soccer World Cup qualification           8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
Soccer World Cup                         1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620

Top 15 teams by total matches played:
home_team
Sweden         1102
England        1091
Argentina      1069
Brazil         1060
Germany        1032
South Korea    1008
Hungary        1004
Mexico         1003
Uruguay         973
France          936
Italy           891
Poland          890
Switzerland     885
Netherlands     880
Norway          873

Matches per decade:
date
1870s      13
1880s      55
1890s      59
1900s     137
1910s     330
1920s     831
1930s    1079
1940s     833
1950s  

# Task 4:Engineer features for the model

In [4]:
import pandas as pd

MAJOR_TOURNAMENTS = {
    "Soccer World Cup",
    "Soccer World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "Copa América",
    "African Cup of Nations",
}

# --- Helper functions (operate on list of (goals_for, goals_against, won) tuples) ---
def winrate(hist):
    if not hist:
        return 0.5
    return sum(h[2] for h in hist) / len(hist)

def goal_avg(hist):
    if not hist:
        return 1.0
    return sum(h[0] for h in hist) / len(hist)

def recent_form(hist):
    last10 = hist[-10:]
    if len(last10) < 10:
        return 0.5
    return sum(h[2] for h in last10) / 10

# --- Filter and sort ---
filtered = (
    matches[matches["date"] >= "1990-01-01"]
    .sort_values("date")
    .reset_index(drop=True)
)

# --- Build features row by row ---
team_history = {}   # team -> list of (goals_for, goals_against, won)
rows = []

for _, row in filtered.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    h_hist = team_history.get(home, [])
    a_hist = team_history.get(away, [])

    # Compute features from history BEFORE this match
    if row["home_score"] > row["away_score"]:
        outcome = 0
    elif row["home_score"] == row["away_score"]:
        outcome = 1
    else:
        outcome = 2

    rows.append({
        "date":                   row["date"],
        "home_team":              home,
        "away_team":              away,
        "team_a_winrate":         winrate(h_hist),
        "team_b_winrate":         winrate(a_hist),
        "team_a_goal_avg":        goal_avg(h_hist),
        "team_b_goal_avg":        goal_avg(a_hist),
        "team_a_recent_form":     recent_form(h_hist),
        "team_b_recent_form":     recent_form(a_hist),
        "is_neutral":             int(row["neutral"]),
        "is_major_tournament":    int(row["tournament"] in MAJOR_TOURNAMENTS),
        "outcome":                outcome,
    })

    # Update history AFTER computing features (prevents leakage)
    h_goals, a_goals = row["home_score"], row["away_score"]
    home_won = 1 if h_goals > a_goals else 0
    away_won = 1 if a_goals > h_goals else 0

    team_history.setdefault(home, []).append((h_goals, a_goals, home_won))
    team_history.setdefault(away, []).append((a_goals, h_goals, away_won))

features_df = pd.DataFrame(rows)

print(features_df.shape)
features_df.head(3)

(32212, 12)


,date,home_team,away_team,team_a_winrate,team_b_winrate,team_a_goal_avg,team_b_goal_avg,team_a_recent_form,team_b_recent_form,is_neutral,is_major_tournament,outcome
0,1990-01-12,Algeria,Mali,0.5,0.5,1.0,1.0,0.5,0.5,1,0,0
1,1990-01-14,Algeria,Cameroon,1.0,0.5,5.0,1.0,0.5,0.5,1,0,0
2,1990-01-17,Greece,Belgium,0.5,0.5,1.0,1.0,0.5,0.5,0,0,0


# Task 5: Split data into training and test sets

In [6]:
feature_cols = [
    "team_a_winrate",
    "team_b_winrate",
    "team_a_goal_avg",
    "team_b_goal_avg",
    "team_a_recent_form",
    "team_b_recent_form",
    "is_neutral",
    "is_major_tournament",
]

# Time-based split: train < 2018-01-01, test >= 2018-01-01
train_mask = features_df["date"] < "2018-01-01"
test_mask  = features_df["date"] >= "2018-01-01"

X_train = features_df.loc[train_mask, feature_cols]
X_test  = features_df.loc[test_mask,  feature_cols]
y_train = features_df.loc[train_mask, "outcome"]
y_test  = features_df.loc[test_mask,  "outcome"]

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)
print("\ny_train class distribution (normalised):")
print(y_train.value_counts(normalize=True).sort_index().to_string())

X_train: (24179, 8)
X_test:  (8033, 8)
y_train: (24179,)
y_test:  (8033,)

y_train class distribution (normalised):
outcome
0    0.487117
1    0.236610
2    0.276273


# Task 6: Train and evaluate the model

In [8]:
%pip install -q scikit-learn

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

# --- Train ---
model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# --- Predict ---
y_pred = model.predict(X_test)

# --- Test accuracy ---
test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_acc * 100:.2f}%")

# --- Baseline: always predict most frequent class in y_train ---
majority_class = y_train.value_counts().idxmax()
baseline_acc = accuracy_score(y_test, [majority_class] * len(y_test))
print(f"Baseline accuracy (always predict class {majority_class}): {baseline_acc * 100:.2f}%")

# --- Confusion matrix ---
labels = ["Home win", "Draw", "Away win"]
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm, index=[f"Actual: {l}" for l in labels],
                         columns=[f"Pred: {l}" for l in labels])
print("\nConfusion matrix:")
print(cm_df.to_string())

# --- Feature importances ---
importances = pd.Series(model.feature_importances_, index=feature_cols)
print("\nFeature importances (descending):")
print(importances.sort_values(ascending=False).to_string())


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Test accuracy: 56.02%
Baseline accuracy (always predict class 0): 47.19%

Confusion matrix:
                  Pred: Home win  Pred: Draw  Pred: Away win
Actual: Home win            3393          32             366
Actual: Draw                1394          31             413
Actual: Away win            1273          55            1076

Feature importances (descending):
team_b_winrate         0.221124
team_a_winrate         0.205628
team_b_goal_avg        0.188185
team_a_goal_avg        0.182613
team_b_recent_form     0.075215
team_a_recent_form     0.075054
is_neutral             0.027426
is_major_tournament    0.024757


# Task 7: Save the model and team statistics

In [9]:
import joblib
import pandas as pd
from pathlib import Path

# --- Create models/ directory ---
Path("models").mkdir(exist_ok=True)

# --- Filter: only World Cup-eligible teams ---
wcq = matches[matches["tournament"] == "Soccer World Cup qualification"]
soccer_teams = set(wcq["home_team"]).union(set(wcq["away_team"]))

# --- Build team_stats from the full matches DataFrame ---
team_stats = {}

all_teams = set(matches["home_team"]).union(set(matches["away_team"]))

for team in all_teams:
    # Skip non-Soccer-World-Cup-eligible teams
    if team not in soccer_teams:
        continue

    home_rows = matches[matches["home_team"] == team]
    away_rows = matches[matches["away_team"] == team]

    total = len(home_rows) + len(away_rows)
    if total < 30:
        continue

    home_wins = (home_rows["home_score"] > home_rows["away_score"]).sum()
    away_wins = (away_rows["away_score"] > away_rows["home_score"]).sum()
    total_wins = int(home_wins + away_wins)

    home_goals = home_rows["home_score"].sum()
    away_goals = away_rows["away_score"].sum()
    total_goals = home_goals + away_goals

    # Recent form: last 10 matches by date
    home_cols = home_rows[["date"]].assign(goals_for=home_rows["home_score"].values,
                                           won=(home_rows["home_score"] > home_rows["away_score"]).astype(int).values)
    away_cols = away_rows[["date"]].assign(goals_for=away_rows["away_score"].values,
                                           won=(away_rows["away_score"] > away_rows["home_score"]).astype(int).values)
    all_team_matches = pd.concat([home_cols, away_cols]).sort_values("date")
    last10 = all_team_matches.tail(10)
    recent = float(last10["won"].mean()) if len(last10) >= 10 else 0.5

    team_stats[team] = {
        "winrate":        total_wins / total,
        "goal_avg":       float(total_goals) / total,
        "recent_form":    recent,
        "matches_played": total,
    }

# --- Save model and team data ---
joblib.dump(model, "models/match_predictor.pkl")
joblib.dump({"team_stats": team_stats, "feature_cols": feature_cols}, "models/team_data.pkl")

print(f"Teams stored: {len(team_stats)}")

# --- Top 5 by winrate among teams with >= 100 matches ---
top5 = (
    pd.DataFrame.from_dict(team_stats, orient="index")
    .query("matches_played >= 100")
    .sort_values("winrate", ascending=False)
    .head(5)[["winrate", "goal_avg", "matches_played"]]
)
print("\nTop 5 teams by win rate (≥100 matches):")
print(top5.to_string())

Teams stored: 215

Top 5 teams by win rate (≥100 matches):
          winrate  goal_avg  matches_played
Brazil   0.632075  2.166981            1060
Spain    0.586735  2.038265             784
Germany  0.578488  2.242248            1032
England  0.571036  2.178735            1091
Iran     0.566069  1.876020             613


# Task 8: Try the model

In [10]:
import pandas as pd

def predict_match(team_a, team_b, is_neutral=True, is_major_tournament=True):
    if team_a not in team_stats:
        raise ValueError(f"Team '{team_a}' not found in team_stats. Check spelling or use a World Cup-eligible team.")
    if team_b not in team_stats:
        raise ValueError(f"Team '{team_b}' not found in team_stats. Check spelling or use a World Cup-eligible team.")

    a = team_stats[team_a]
    b = team_stats[team_b]

    row = pd.DataFrame([{
        "team_a_winrate":      a["winrate"],
        "team_b_winrate":      b["winrate"],
        "team_a_goal_avg":     a["goal_avg"],
        "team_b_goal_avg":     b["goal_avg"],
        "team_a_recent_form":  a["recent_form"],
        "team_b_recent_form":  b["recent_form"],
        "is_neutral":          int(is_neutral),
        "is_major_tournament": int(is_major_tournament),
    }])[feature_cols]   # reindex to guarantee column order

    proba = model.predict_proba(row)

    return {
        "team_a_win_prob": float(proba[0][0]),
        "draw_prob":       float(proba[0][1]),
        "team_b_win_prob": float(proba[0][2]),
    }


result1 = predict_match("Brazil", "Argentina")
print("Brazil vs Argentina:")
print(f"  Brazil win:    {result1['team_a_win_prob']:.1%}")
print(f"  Draw:          {result1['draw_prob']:.1%}")
print(f"  Argentina win: {result1['team_b_win_prob']:.1%}")

print()

result2 = predict_match("Germany", "Brazil")
print("Germany vs Brazil:")
print(f"  Germany win: {result2['team_a_win_prob']:.1%}")
print(f"  Draw:        {result2['draw_prob']:.1%}")
print(f"  Brazil win:  {result2['team_b_win_prob']:.1%}")

Brazil vs Argentina:
  Brazil win:    39.2%
  Draw:          20.9%
  Argentina win: 39.9%

Germany vs Brazil:
  Germany win: 61.8%
  Draw:        14.5%
  Brazil win:  23.8%


# Task 9: Create an interactive UI application

In [11]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

st.set_page_config(
    page_title="Soccer 2026 Match Predictor",
    page_icon="⚽",
    layout="centered",
)


@st.cache_resource
def load_artifacts():
    model = joblib.load(Path("models/match_predictor.pkl"))
    team_data = joblib.load(Path("models/team_data.pkl"))
    return model, team_data["team_stats"], team_data["feature_cols"]


model, team_stats, feature_cols = load_artifacts()

st.title("⚽ Soccer 2026 Match Predictor")
st.caption("Predictions based on historical international football results.")

team_names = sorted(team_stats.keys())

col1, col2 = st.columns(2)
with col1:
    default_a = team_names.index("Brazil") if "Brazil" in team_names else 0
    team_a = st.selectbox("Team A", team_names, index=default_a)
with col2:
    default_b = team_names.index("Argentina") if "Argentina" in team_names else 1
    team_b = st.selectbox("Team B", team_names, index=default_b)

is_neutral = st.checkbox("Neutral venue", value=True)
is_major = st.checkbox("Major tournament (e.g. World Cup)", value=True)

if st.button("Predict", type="primary", use_container_width=True):
    if team_a == team_b:
        st.error("Please pick two different teams.")
    else:
        a = team_stats[team_a]
        b = team_stats[team_b]

        row = pd.DataFrame([{
            "team_a_winrate":      a["winrate"],
            "team_b_winrate":      b["winrate"],
            "team_a_goal_avg":     a["goal_avg"],
            "team_b_goal_avg":     b["goal_avg"],
            "team_a_recent_form":  a["recent_form"],
            "team_b_recent_form":  b["recent_form"],
            "is_neutral":          int(is_neutral),
            "is_major_tournament": int(is_major),
        }])[feature_cols]

        proba = model.predict_proba(row)[0]
        p_a, p_draw, p_b = float(proba[0]), float(proba[1]), float(proba[2])

        st.subheader("Predicted probabilities")
        m1, m2, m3 = st.columns(3)
        m1.metric(f"{team_a} wins", f"{p_a:.1%}")
        m2.metric("Draw",           f"{p_draw:.1%}")
        m3.metric(f"{team_b} wins", f"{p_b:.1%}")

        st.progress(p_a,    text=f"{team_a} wins — {p_a:.1%}")
        st.progress(p_draw, text=f"Draw — {p_draw:.1%}")
        st.progress(p_b,    text=f"{team_b} wins — {p_b:.1%}")

        st.subheader("Team statistics")
        stats_table = pd.DataFrame(
            {
                "Win rate":          [f"{a['winrate']:.1%}",       f"{b['winrate']:.1%}"],
                "Avg goals scored":  [f"{a['goal_avg']:.2f}",      f"{b['goal_avg']:.2f}"],
                "Recent form (10)": [f"{a['recent_form']:.1%}",   f"{b['recent_form']:.1%}"],
                "Matches played":    [a["matches_played"],          b["matches_played"]],
            },
            index=[team_a, team_b],
        )
        st.table(stats_table)

Writing app.py


# Task 10: Launch the UI application

In [12]:
import subprocess
import sys
import time
import webbrowser

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.headless", "true",
        "--server.port", "8501",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("Starting Streamlit server...")
time.sleep(4)

webbrowser.open("http://localhost:8501")

print("App running at http://localhost:8501")
print("To stop the server, run: streamlit_process.terminate()")

Starting Streamlit server...
App running at http://localhost:8501
To stop the server, run: streamlit_process.terminate()
